# Estudio 06 · Paso b de clima · Lluvia de CHIRPS v3 (b1 y b2)

Preregistro `STEP_B` (commit `b82bf80`, aprobado por Rosa tal cual el 2026-11-06, con el cruce b2 frente a b1 y empate
a favor de b1). Fuente: CHIRPS v3 diario `sat`, versión final de Google Earth Engine, media del polígono distrital
ponderada por área; silver `data/silver/precip_chirps_v3_semanal_desde_2000.csv` (90 675 filas, 2000-01-02 a
2026-09-20, todo `final`).

- **b1** = L + 5a + `precip_total_mm_media5_h2` (lluvia CHIRPS, definición de la A3 de la V2).
- **b2** = L + 5a + `hum_rel_media_media5_h2` (humedad ERA5-Land de la 1b′) + lluvia CHIRPS.

Cada una frente a L + 5a con la regla de 5 temporadas (la de la 1b′); se adopta con «gana» en >= 2 de 3 horizontes
y sin «pierde» ni «compensa». Disponibilidad: una semana se usa solo si su `fecha_disponible` es <= `origen_cierre`
+ 14 días (referencia de Rosa, *hipótesis*).

In [1]:
import json
import sys

import numpy as np
import pandas as pd

from denguekay.config.paths import GOLD, RESEARCH
from denguekay.evaluation import episodes as diag
from denguekay.evaluation import protocol as pa
from denguekay.model import probabilistic_output as sp
from denguekay.model import tracking
from denguekay.model.device import xgboost_settings

STUDY = RESEARCH / "06_model_improvement"
sys.path.insert(0, str(STUDY / "code"))
import phase0_diagnosis as p0
import phase1_history as p1
import phase1b_prime as pb
import phase2_calibration as p2
import phase3_onsets as p3
import phase5_counts as p5
import phase6_other_families as p6
import phase_b_rain as pbr
import preregistration as pr

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
METRICS = STUDY / "docs" / "metricas" / "paso_b.json"
OUTPUT_DIR = p3.OUTPUT_DIR.parent / "paso_b"
device = xgboost_settings()
inp = p3.phase3_inputs()
chirps = pbr.read_chirps()
humidity = pb.era5_land_panel(inp["panel"])
origins = pbr.add_rain(inp["origenes"], pbr.rain_by_origin_week(chirps))
origins = pb.add_humidity(origins, pb.humidity_by_origin_week(humidity))
print(len(chirps), chirps.version.value_counts().to_dict(), chirps.semana_inicio.min().date(), chirps.semana_inicio.max().date())
device

90675 {'final': 90675} 2000-01-02 2026-09-20


{'device': 'cuda', 'n_jobs': 12}

## 1 · Guardas G25–G29 y, para b2, G17 y G19 (antes de mirar resultados; G26 es el test de no fuga)

In [2]:
gold = pd.read_csv(GOLD / "meteo_socio_epi_piura_2017_2025_h2_gold.csv", dtype={"ubigeo": str},
                   float_precision="round_trip")
g25 = pbr.guard_definition(gold, inp["panel"])
g27 = pbr.guard_availability(origins, pr.STEP_B_REFERENCE_DAYS)
g28 = {v: pbr.guard_coverage(origins, pr.STEP_B_COLUMNS[v]) for v in ("b1", "b2")}
base = p6.l_plus_5a(inp["serie"])
saved5 = json.loads((STUDY / "docs" / "metricas" / "fase5.json").read_text(encoding="utf-8"))
g29 = p6.guard_g16_reproduction(base["evaluaciones"], saved5["veredictos"]["5a"]["metricas"])
g17 = pb.guard_g17(gold, inp["panel"])
g19 = pb.guard_g19(origins)
versions = pbr.versions_used(origins[pd.to_datetime(origins.origen_cierre) >= pd.Timestamp("2005-01-01")], chirps)
guards = {"G25": g25, "G27": g27, "G28": g28, "G29": g29, "G17": g17, "G19": g19}
ok = g25["pasa"] and g27["pasa"] and all(x["pasa"] for x in g28.values()) and g29["pasa"] and g17["pasa"] and g19["pasa"]
if not ok:
    raise RuntimeError(f"Falla una guarda: se detiene el paso b. {guards}")
print("versiones de la última semana usada (orígenes desde 2005):", versions)
guards

versiones de la última semana usada (orígenes desde 2005): {'final': 71110}


{'G25': {'filas_gold': 30485,
  'con_valor': 30095,
  'mismos_vacios': True,
  'diferencia_maxima': 0.0,
  'pasa': True},
 'G27': {'referencia': 'origen_cierre + 14 días (hipótesis)',
  'origenes_con_lluvia': 71175,
  'sin_fecha_disponible': 0,
  'afectados': 0,
  'afectados_contra_origen_cierre': 71175,
  'margen_minimo_dias': 4,
  'pasa': True},
 'G28': {'b1': {'origenes_desde_2005': 71110, 'sin_valor': 0, 'pasa': True},
  'b2': {'origenes_desde_2005': 71110, 'sin_valor': 0, 'pasa': True}},
 'G29': {'diferencia_maxima': 4.9451664985156185e-11,
  'filas': 18,
  'pasa': True},
 'G17': {'filas_gold': 30485,
  'con_valor': 30095,
  'mismos_vacios': True,
  'diferencia_maxima': 0.0,
  'pasa': True},
 'G19': {'origenes_desde_2005': 71110, 'sin_H': 0, 'pasa': True}}

## 2 · Variantes b1 y b2 (5 semillas)

In [3]:
base_columns, hp = p3._config()
blocks = p1.seasonal_blocks(p1.FIRST_BLOCK_SEASON_LONG)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
cands = {}
for variant in ("b1", "b2"):
    columns = [*base_columns, *pr.STEP_B_COLUMNS[variant]]
    preds, evals, fits = {}, {}, []
    for seed in pr.SEEDS:
        pred = p3.predict_blocks_weighted(inp["matrices"], origins, blocks, seed, columns, hp)
        fits.append(pred.attrs.pop("ajustes").assign(semilla=seed))
        preds[seed] = pred
        evals[seed] = p5.evaluate_mixture(pred, inp["serie"])
    cands[variant] = {"predicciones": preds, "evaluaciones": evals}
    pd.concat(preds.values(), ignore_index=True).to_csv(OUTPUT_DIR / f"predicciones_{variant}.csv.gz", index=False)
    pd.concat(fits, ignore_index=True).drop(columns=["corte"]).to_csv(OUTPUT_DIR / f"ajustes_{variant}.csv", index=False)
    print(variant, columns)

b1 ['hist_lag_a', 'hist_lag_b', 'hist_media_4', 'hist_semanas_pos_4', 'semana_epi_seno', 'semana_epi_coseno', 'propia_semanas_desde_ultimo_caso', 'propia_pendiente_4', 'propia_log_casos_temporada_anterior', 'precip_total_mm_media5_h2']


b2 ['hist_lag_a', 'hist_lag_b', 'hist_media_4', 'hist_semanas_pos_4', 'semana_epi_seno', 'semana_epi_coseno', 'propia_semanas_desde_ultimo_caso', 'propia_pendiente_4', 'propia_log_casos_temporada_anterior', 'hum_rel_media_media5_h2', 'precip_total_mm_media5_h2']


## 3 · Veredicto preregistrado (cada variante frente a L + 5a; cruce si se adoptan las dos)

In [4]:
decs, adopted = {}, {}
for variant in ("b1", "b2"):
    decs[variant] = p3.compare(cands[variant], base)
    adopted[variant] = bool(pr.adopt_step_b(decs[variant]["veredicto"]))
    print(variant, "| adoptada:", adopted[variant])
    display(decs[variant]["veredicto"])
cross = None
if adopted["b1"] and adopted["b2"]:
    cross = p3.compare(cands["b2"], cands["b1"])
    winner = "b2" if pr.adopt_step_b(cross["veredicto"]) else "b1"
    display(cross["veredicto"])
else:
    winner = next((v for v in pr.STEP_B_SIMPLICITY if adopted[v]), None)
print("Base nueva:", f"L + 5a + {winner}" if winner else "sigue L + 5a")

b1 | adoptada: False


,horizonte,veredicto,mejora_antelacion,empeora_antelacion,vetos,vetos_contra_referencia,a_favor_f1_050,a_favor_auprc,a_favor_brier,a_favor_razon_mae,en_contra_f1_050,en_contra_auprc,en_contra_brier,en_contra_razon_mae,a_favor_antelacion,en_contra_antelacion
0,2,empate,False,False,None,razon_mae,0,2,1,3,0,0,0,1,0,0
1,3,empate,False,False,None,"auprc, razon_mae",0,3,1,3,0,0,0,1,0,0
2,4,empate,False,False,None,"auprc, razon_mae",1,3,2,4,0,0,0,1,0,0


b2 | adoptada: False


,horizonte,veredicto,mejora_antelacion,empeora_antelacion,vetos,vetos_contra_referencia,a_favor_f1_050,a_favor_auprc,a_favor_brier,a_favor_razon_mae,en_contra_f1_050,en_contra_auprc,en_contra_brier,en_contra_razon_mae,a_favor_antelacion,en_contra_antelacion
0,2,empate,False,False,None,NaN,0,1,0,2,0,0,1,1,0,0
1,3,empate,False,False,None,razon_mae,0,2,2,3,0,0,1,1,0,0
2,4,empate,False,False,None,"brier, razon_mae",1,2,3,3,0,0,0,1,0,0


Base nueva: sigue L + 5a


In [5]:
pd.concat({v: d["antelacion"] for v, d in decs.items()}, names=["variante"]).round(3)

horizonte  episodios  diferencia  ic95_inferior  ic95_superior  diferencia_temporada_2010  diferencia_temporada_2015  diferencia_temporada_2016  diferencia_temporada_2022  diferencia_temporada_2023  \
variante                                                                                                                                                                                                            
b1       0          2        193       0.005         -0.021          0.031                      0.033                      0.000                        0.0                      0.000                      0.000   
         1          3        193      -0.010         -0.036          0.016                      0.000                      0.000                        0.0                     -0.026                     -0.018   
         2          4        193      -0.010         -0.026          0.000                      0.000                      0.000                        0.0                     -0.026                     -0.018   
b2       0          2        193      -0.005         -0.031          0.016                      0.000                     -0.029                        0.0                      0.000                      0.000   
         1          3        193      -0.005         -0.026          0.016                      0.000                      0.000                        0.0                     -0.026                      0.000   
         2          4        193      -0.005         -0.016          0.000                      0.000                      0.000                        0.0                     -0.026                      0.000   

            episodios_temporada_2010  episodios_temporada_2015  episodios_temporada_2016  episodios_temporada_2022  episodios_temporada_2023  
variante                                                                                                                                      
b1       0                        30                        34                        34                        38                        57  
         1                        30                        34                        34                        38                        57  
         2                        30                        34                        34                        38                        57  
b2       0                        30                        34                        34                        38                        57  
         1                        30                        34                        34                        38                        57  
         2                        30                        34                        34                        38                        57

In [6]:
cols = ["horizonte", "bloque", "positivos_cand", "f1_050_cand", "f1_050_ref", "auprc_cand", "auprc_ref",
        "brier_cand", "brier_ref", "razon_mae_cand", "razon_mae_ref", "razon_rmse_cand", "razon_rmse_ref"]
pd.concat({v: d["metricas"][cols] for v, d in decs.items()}, names=["variante"]).round(3)

horizonte          bloque  positivos_cand  f1_050_cand  f1_050_ref  auprc_cand  auprc_ref  brier_cand  brier_ref  razon_mae_cand  razon_mae_ref  razon_rmse_cand  razon_rmse_ref
variante                                                                                                                                                                                     
b1       3           2  temporada_2010           234.0        0.723       0.722       0.782      0.780       0.030      0.030           0.886          0.907            0.894           0.899
         8           2  temporada_2015           557.0        0.772       0.772       0.886      0.885       0.051      0.051           0.896          0.889            0.929           0.932
         9           2  temporada_2016           395.0        0.657       0.657       0.757      0.757       0.059      0.059           1.021          1.030            0.983           0.995
         15          2  temporada_2022           431.0        0.715       0.712       0.779      0.781       0.054      0.054           0.944          0.946            0.949           0.965
         16          2  temporada_2023          1039.0        0.859       0.857       0.948      0.947       0.061      0.062           0.778          0.808            0.843           0.893
         17          2  temporada_2024           857.0        0.805       0.799       0.894      0.893       0.072      0.074           0.903          1.136            0.901           1.316
         21          3  temporada_2010           234.0        0.638       0.631       0.710      0.706       0.036      0.036           0.886          0.891            0.911           0.890
         26          3  temporada_2015           557.0        0.650       0.649       0.842      0.839       0.064      0.064           0.839          0.828            0.906           0.914
         27          3  temporada_2016           395.0        0.628       0.627       0.709      0.709       0.065      0.065           1.032          1.042            0.996           1.016
         33          3  temporada_2022           431.0        0.706       0.704       0.769      0.771       0.056      0.056           0.903          0.927            0.946           0.990
         34          3  temporada_2023          1039.0        0.833       0.835       0.932      0.930       0.072      0.073           0.734          0.763            0.798           0.852
         35          3  temporada_2024           857.0        0.792       0.788       0.879      0.876       0.078      0.079           0.860          1.238            0.868           1.485
         39          4  temporada_2010           234.0        0.463       0.476       0.607      0.600       0.042      0.042           0.865          0.876            0.881           0.859
         44          4  temporada_2015           557.0        0.477       0.480       0.814      0.811       0.075      0.075           0.803          0.793            0.875           0.880
         45          4  temporada_2016           395.0        0.595       0.590       0.660      0.660       0.069      0.070           1.054          1.062            1.016           1.033
         51          4  temporada_2022           431.0        0.649       0.652       0.741      0.743       0.060      0.060           0.899          0.929            0.967           1.038
         52          4  temporada_2023          1039.0        0.814       0.811       0.924      0.922       0.080      0.081           0.701          0.733            0.768           0.815
         53          4  temporada_2024           857.0        0.769       0.761       0.849      0.840       0.086      0.089           0.844          1.362            0.888           1.690
b2       3           2  temporada_2010           234.0        0.707       0.722       0.781      0.780       0.030      0.030           0.887          0.907            0.920           0.899
         8           2 

## 4 · Lecturas que no deciden

In [7]:
no2010 = pd.concat({v: d["lectura_sin_2010"][["horizonte", "veredicto", "vetos"]] for v, d in decs.items()}, names=["variante"])
mae_rules = {v: p5.compare(cands[v], base) for v in ("b1", "b2")}
mae = pd.concat({v: m["veredicto"][["horizonte", "veredicto", "vetos"]] for v, m in mae_rules.items()}, names=["variante"])
contrast = pd.concat({v: d["metricas"][d["metricas"].bloque.eq(pr.block_name(pr.CONTRAST_SEASON))][cols]
                      for v, d in decs.items()}, names=["variante"])
display(no2010)
display(mae)
contrast.round(3)

horizonte veredicto vetos
variante                             
b1       0          2    empate  None
         1          3    empate  None
         2          4    empate  None
b2       0          2    empate  None
         1          3    empate  None
         2          4    empate  None

horizonte veredicto vetos
variante                             
b1       0          2    empate  None
         1          3    empate  None
         2          4    empate  None
b2       0          2    empate  None
         1          3    empate  None
         2          4    empate  None

horizonte          bloque  positivos_cand  f1_050_cand  f1_050_ref  auprc_cand  auprc_ref  brier_cand  brier_ref  razon_mae_cand  razon_mae_ref  razon_rmse_cand  razon_rmse_ref
variante                                                                                                                                                                                     
b1       17          2  temporada_2024           857.0        0.805       0.799       0.894      0.893       0.072      0.074           0.903          1.136            0.901           1.316
         35          3  temporada_2024           857.0        0.792       0.788       0.879      0.876       0.078      0.079           0.860          1.238            0.868           1.485
         53          4  temporada_2024           857.0        0.769       0.761       0.849      0.840       0.086      0.089           0.844          1.362            0.888           1.690
b2       17          2  temporada_2024           857.0        0.805       0.799       0.894      0.893       0.072      0.074           0.899          1.136            0.890           1.316
         35          3  temporada_2024           857.0        0.793       0.788       0.880      0.876       0.077      0.079           0.856          1.238            0.862           1.485
         53          4  temporada_2024           857.0        0.767       0.761       0.850      0.840       0.086      0.089           0.842          1.362            0.881           1.690

In [8]:
variants = {"L + 5a": base, "b1": cands["b1"], "b2": cands["b2"]}
rd = {k: p2.readings(x, inp["serie"]) for k, x in variants.items()}
recall = pd.concat({k: x["recall_por_tipo"] for k, x in rd.items()}, names=["variante"]).reset_index(level=0)
deciding = [pr.block_name(s) for s in pr.DECIDING_SEASONS]
new = recall[recall.tipo.eq("brote_nuevo") & recall.bloque.isin(deciding)]
new_wide = new.pivot_table(index=["horizonte", "bloque"], columns="variante", values="recall")
new_wide["semanas"] = new[new.variante.eq("L + 5a")].set_index(["horizonte", "bloque"]).semanas
for v in ("b1", "b2"):
    d = new_wide[v] - new_wide["L + 5a"]
    print(f"Recall de brotes nuevos {v}: mejora en", int((d > 0).sum()), "celdas, empeora en", int((d < 0).sum()), "de", len(d))
new_wide.round(3)

Recall de brotes nuevos b1: mejora en 1 celdas, empeora en 3 de 15
Recall de brotes nuevos b2: mejora en 1 celdas, empeora en 5 de 15


variante                  L + 5a     b1     b2  semanas
horizonte bloque                                       
2         temporada_2010   0.118  0.118  0.078     51.0
          temporada_2015   0.067  0.067  0.067     60.0
          temporada_2016   0.132  0.132  0.132     53.0
          temporada_2022   0.122  0.122  0.122     49.0
          temporada_2023   0.174  0.174  0.174     86.0
3         temporada_2010   0.059  0.059  0.059     51.0
          temporada_2015   0.033  0.033  0.017     60.0
          temporada_2016   0.113  0.094  0.094     53.0
          temporada_2022   0.163  0.122  0.122     49.0
          temporada_2023   0.186  0.198  0.198     86.0
4         temporada_2010   0.020  0.020  0.020     51.0
          temporada_2015   0.017  0.017  0.017     60.0
          temporada_2016   0.057  0.057  0.057     53.0
          temporada_2022   0.122  0.102  0.102     49.0
          temporada_2023   0.174  0.174  0.174     86.0

In [9]:
# Frente a la V2 en 2022 y 2023 (predicciones guardadas del estudio 05; su Platt y su clima son los suyos)
v2_evals = p0.evaluate_seeds(p0.v2_predictions(), inp["serie"])
med = {k: pa.seed_median([e["metricas"] for e in x["evaluaciones"].values()]) for k, x in variants.items()}
med = {"V2": pa.seed_median([e["metricas"] for e in v2_evals.values()]), **med}
keep = ["horizonte", "bloque", "positivos", "f1_050", "recall_050", "auprc", "brier", "razon_mae", "razon_rmse"]
v2_table = pd.concat({k: m[m.bloque.isin(["temporada_2022", "temporada_2023"])][keep] for k, m in med.items()},
                     names=["variante"]).reset_index(level=0)
hu = pd.concat({k: sp.hu0007_4_compliance(m[m.bloque.isin(["temporada_2022", "temporada_2023"])]) for k, m in med.items()},
               names=["variante"]).reset_index(level=0)
display(v2_table.round(3))
hu

,variante,horizonte,bloque,positivos,f1_050,recall_050,auprc,brier,razon_mae,razon_rmse
2,V2,2,temporada_2022,431.0,0.709,0.687,0.782,0.054,0.919,0.938
3,V2,2,temporada_2023,1039.0,0.858,0.821,0.948,0.063,0.818,0.869
7,V2,3,temporada_2022,431.0,0.594,0.462,0.773,0.059,0.883,0.922
8,V2,3,temporada_2023,1039.0,0.828,0.768,0.930,0.075,0.747,0.818
12,V2,4,temporada_2022,431.0,0.333,0.202,0.741,0.068,0.838,0.905
13,V2,4,temporada_2023,1039.0,0.811,0.733,0.920,0.084,0.708,0.779
15,L + 5a,2,temporada_2022,431.0,0.712,0.696,0.781,0.054,0.946,0.965
16,L + 5a,2,temporada_2023,1039.0,0.857,0.828,0.947,0.062,0.808,0.893
33,L + 5a,3,temporada_2022,431.0,0.704,0.682,0.771,0.056,0.927,0.990
34,L + 5a,3,temporada_2023,1039.0,0.835,0.794,0.930,0.073,0.763,0.852


,variante,horizonte,bloque,precision_050,recall_050,f1_050,razon_mae,razon_rmse,cumple_precision_050,cumple_recall_050,cumple_f1_050,cumple_razon_mae,cumple_razon_rmse,cumple_clasificacion,cumple_regresion
2,V2,2,temporada_2022,0.734491,0.686775,0.709288,0.918743,0.937701,True,False,True,False,False,False,False
3,V2,2,temporada_2023,0.898841,0.820982,0.858149,0.818114,0.868532,True,True,True,True,False,True,False
7,V2,3,temporada_2022,0.835391,0.461717,0.594030,0.883139,0.921506,True,False,False,False,False,False,False
8,V2,3,temporada_2023,0.896745,0.768046,0.827801,0.747036,0.818020,True,False,True,True,True,False,True
12,V2,4,temporada_2022,0.959596,0.201856,0.333333,0.838328,0.905236,True,False,False,True,False,False,False
13,V2,4,temporada_2023,0.911483,0.733397,0.811129,0.707967,0.779131,True,False,True,True,True,False,True
15,L + 5a,2,temporada_2022,0.728155,0.696056,0.711905,0.945681,0.964720,True,False,True,False,False,False,False
16,L + 5a,2,temporada_2023,0.888545,0.827719,0.857428,0.807630,0.892501,True,True,True,True,False,True,False
33,L + 5a,3,temporada_2022,0.727723,0.682135,0.704327,0.927262,0.989964,True,False,True,False,False,False,False
34,L + 5a,3,temporada_2023,0.880342,0.794033,0.835020,0.762822,0.852290,True,False,True,True,False,False,False


In [10]:
# CHIRPS − «Best match» de precip_total_mm_media5_h2, 2017-2025 por año (65 distritos, por semana de origen)
current = inp["panel"][["ubigeo", "semana_inicio", "precip_total_mm"]].copy()
current["semana_inicio"] = pd.to_datetime(current.semana_inicio)
current["ubigeo"] = current.ubigeo.astype(str)
bm = pbr.rain_by_origin_week(current)[["ubigeo", "semana_origen", "R"]]
ch = pbr.rain_by_origin_week(chirps)[["ubigeo", "semana_origen", "R"]]
both = bm.merge(ch, on=["ubigeo", "semana_origen"], suffixes=("_best_match", "_chirps")).dropna()
both["anio"] = both.semana_origen.dt.year
both["dif"] = both.R_chirps - both.R_best_match
rain_diff = both.groupby("anio").apply(lambda d: pd.Series({
    "origenes": len(d), "media_best_match": d.R_best_match.mean(), "media_chirps": d.R_chirps.mean(),
    "diferencia_media": d.dif.mean(), "mae": d.dif.abs().mean(), "correlacion": d.R_chirps.corr(d.R_best_match)}),
    include_groups=False)
rain_diff.round(3)

,origenes,media_best_match,media_chirps,diferencia_media,mae,correlacion
anio,,,,,,
2017,3185.0,35.646,22.564,-13.082,15.458,0.895
2018,3380.0,12.676,3.918,-8.758,8.998,0.769
2019,3380.0,18.764,7.449,-11.316,12.104,0.815
2020,3380.0,11.640,4.184,-7.456,7.976,0.794
2021,3380.0,12.544,5.739,-6.805,7.688,0.789
2022,3380.0,11.527,4.774,-6.754,7.342,0.854
2023,3445.0,27.135,15.117,-12.019,13.387,0.855
2024,3380.0,13.524,4.615,-8.909,9.406,0.796
2025,3315.0,18.526,6.864,-11.662,12.272,0.806


## 5 · Salidas: métricas y MLflow

In [11]:
to_json = lambda d: {k: diag.to_json(x) for k, x in d.items() if isinstance(x, pd.DataFrame)}
content = {
    "dispositivo": device, "guardas": guards, "versiones_usadas": versions,
    "veredictos": {v: to_json(d) for v, d in decs.items()}, "adoptadas": adopted,
    "cruce_b2_frente_a_b1": to_json(cross) if cross else None, "base_resultante": f"L + 5a + {winner}" if winner else "L + 5a",
    "lectura_sin_2010": diag.to_json(no2010.reset_index(level=0)), "lectura_regla_mae": diag.to_json(mae.reset_index(level=0)),
    "contraste_2024": diag.to_json(contrast.reset_index(level=0)),
    "recall_brotes_nuevos": diag.to_json(new_wide.reset_index()), "frente_a_v2": diag.to_json(v2_table),
    "hu0007_4_2022_2023": diag.to_json(hu), "lluvia_chirps_menos_best_match": diag.to_json(rain_diff.reset_index()),
}
content = json.loads(json.dumps(content, default=lambda o: o.item() if hasattr(o, "item") else str(o)))
for section, value in content.items():
    diag.save_metrics_section(METRICS, section, value)
verdicts = pd.concat({v: d["veredicto"] for v, d in decs.items()}, names=["variante"]).reset_index(level=0)
run_id = tracking.log_study(
    "mejora_modelo_paso_b_lluvia_chirps", pd.DataFrame(columns=["numero", "estado"]),
    parent={"params": {"fase": "paso_b", "preregistro": "b82bf80", "adoptada_b1": str(adopted["b1"]),
                       "adoptada_b2": str(adopted["b2"]), "base_resultante": content["base_resultante"],
                       "veredicto": "; ".join(f"{r.variante} h{r.horizonte}: {r.veredicto}" for r in verdicts.itertuples()),
                       **{f"xgb.{k}": str(x) for k, x in device.items()}},
            "metricas": {}},
    artifacts=[METRICS], tables={"veredicto": verdicts},
    description="Paso b de clima del estudio 06: lluvia CHIRPS v3 (b1) y humedad ERA5-Land + lluvia CHIRPS (b2) sobre L + 5a.")
run_id

C:\Users\Rosa\RufoEsEternoElRegreso\Denguekay\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MLflow: estudio c2c52c15a2bf4e71a5d7b339e51c04e6 en 'denguekay/mejora_modelo_paso_b_lluvia_chirps' (sqlite:///C:/Users/Rosa/RufoEsEternoElRegreso/Denguekay/mlflow.db)


'c2c52c15a2bf4e71a5d7b339e51c04e6'